In [1]:
pip install langchain[openai]

   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 3.7 MB/s  0:00:00
   ---------------------------------------- 0.0/744.6 kB ? eta -:--:--
   ---------------------------------------- 744.6/744.6 kB 4.7 MB/s  0:00:00
   ---------------------------------------- 0.0/988.0 kB ? eta -:--:--
   ------------------------------- -------- 786.4/988.0 kB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 988.0/988.0 kB 4.4 MB/s  0:00:00

   - --------------------------------------  1/22 [xxhash]
   ----- ----------------------------------  3/22 [regex]
   ------- --------------------------------  4/22 [pyyaml]
   ------- --------------------------------  4/22 [pyyaml]
   ------- --------------------------------  4/22 [pyyaml]
   ------------ ---------------------------  7/22 [langchain-protocol]
   ---------------- -----------------------  9/22 [httpcore]
   ---------------- -----------------------  9/22

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['OPENAI_API_KEY'] = os.get_env('OPENAI_API_KEY')


In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4o-mini", model_provider="openai")
result = model.invoke("What is 3 plus 7 ?")
result

In [ ]:
from IPython.display import display, Markdown
display(Markdown(result.content))

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke("tell me about Mahatma Gandhi in 3 bullet points.")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [SystemMessage(content="Translatethe following from English to German"),
            HumanMessage(content="Hello!")]

result =llm.invoke(messages)
result 

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
parser = StrOutputParser()
parser.invoke(result)

In [ ]:
chain = llm | parser
chain.invoke(messages)
#All chains must be object of Runnables 

In [ ]:
from langchain_core.prompts import PromptTemplate
template = PromptTemplate(
    template = "Translate the following from English into {language}:{text}"
)
template
template.invoke({"language":"Hindi","text":"Hello!"})


In [ ]:
chain = template|llm|parser
chain.invoke({"language":"Hindi","text":"Hello!"})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following from English to {language}"),
    ("human", "{text}")
])

In [ ]:
chain = chat_template | llm | parser
chain.invoke({"language":"Hindi","text":"Hello!"})

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name:str = Field(description="Name of the person")
    age:int = Field(description="Age of the person")
    city:str = Field(description="City of the person")

parser = PydanticOutputParser(pydantic_object=Person)
parser.get_format_instructions()
parser.invoke(result)


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user query. Wrap the output in json tags\n {format_instructions}"),
        ("human", "{query}")
]).partial(format_instructions=parser.get_format_instructions())

In [ ]:
chain = prompt | llm | parser
chain.invoke({"query":"What is the name, age and city of Elon Musk ?"}) # gives op in Person object
chain = prompt | llm | StrOutputParser()
chain.invoke({"query":"What is the name, age and city of Elon Musk ?"}) #gives op in json 